# GAP 3 — Decision-Centric Explainability

**Objective 3: Generate Decision-Centric Explainable AI Outputs**

> Generate decision-centric, counterfactual explanations that clearly show how financial
> decisions causally influence long-term outcomes beyond static post-hoc interpretations.

Objectives 1 and 2 already produce everything a decision needs: a detected bias with a
confidence, causal effect estimates, SHAP attributions, and a 10–30 year Monte Carlo
comparison of holding versus selling.

None of that is readable by the person whose money it is.

This notebook demonstrates the explainability layer that closes the gap — one plain-language
story, in **English or Hindi**, that any user can follow.

### What this notebook shows
1. The problem: what the system said *before* this layer
2. The six-section explanation, end to end
3. All four bias narratives, in both languages
4. The jargon translation layer
5. Validation metrics (readability, completeness, actionability, personalisation, jargon)
6. The downloadable report
7. How the optional LLM layer is prevented from inventing numbers


In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

import explanation_report as report_mod
import explanation_templates as T
from explainer import (DecisionExplainer, explain_bias_only, find_jargon,
                       generate_complete_explanation, render_markdown)

from IPython.display import HTML, Markdown, display

print("Languages     :", {c: T.LANG_NAMES[c] for c in T.SUPPORTED_LANGS})
print("Sections      :", T.SECTIONS)
print("Report formats:", {l: report_mod.available_formats(l) for l in T.SUPPORTED_LANGS})

Languages     : {'en': 'English', 'hi': 'हिंदी'}
Sections      : ['headline', 'whats_happening', 'two_futures', 'money_story', 'why_we_say_this', 'what_to_do_now']
Report formats: {'en': ['md', 'html', 'pdf'], 'hi': ['md', 'html']}


---
## 1. Run the real pipeline

We use a genuine case: **user 42**, whose investment is down 15%, is thinking of selling.
Objective 1 detects the bias; Objective 2 simulates both futures over the user's real horizon.

In [2]:
from integration import analyze_decision_realtime
from simulator import get_simulator

user_input = {
    "user_id"            : 42,
    "unrealized_gl"      : -0.15,
    "crowd_ratio"        : 0.72,
    "age"                : 38,
    "investment_horizon" : 18,
    "risk_tolerance"     : 0.35,
    "monthly_income"     : 60_000,
    "has_emergency_fund" : 1,
    "decision_intent"    : "Thinking of selling",
}

bias_result = analyze_decision_realtime(user_input)
sim_result  = get_simulator().compare_hold_vs_sell(
    user_id=42, current_wealth=75_000, unrealized_gl=-0.15, n_simulations=1_000)

print("Objective 1 →", bias_result["bias_type"],
      f"({bias_result['confidence']:.0%} confidence)")
print("Objective 2 →", sim_result["comparison"]["better_decision"],
      f"| opportunity cost Rs.{abs(sim_result['comparison']['opportunity_cost_inr']):,.0f}")

Objective 1 → loss_aversion (90% confidence)
Objective 2 → HOLD | opportunity cost Rs.16,377


---
## 2. The problem — what the user saw *before* this layer

This is the raw output of Objectives 1 and 2. It is accurate, and almost nobody outside
this project can act on it.

In [3]:
print("ONE-LINE SUMMARY (integration.py)")
print(" ", bias_result["explanation"])
print()
print("CAUSAL CHAIN (integration.py)")
for step in bias_result["causal_chain"]:
    print("  -", step)
print()
print("SHAP CONTRIBUTIONS (bias_detector.py)")
for row in bias_result.get("shap_explanation", [])[:5]:
    print("  -", row)
print()
print("SIMULATION (simulator.py)")
for key in ("mean", "median", "p10", "p90"):
    print(f"  hold {key:<7}: {sim_result['hold'][key]:>14,.0f}")

ONE-LINE SUMMARY (integration.py)
  Detected Loss Aversion bias (confidence 90%, risk 44%). 💡 Recommended intervention: Reframe losses (cognitive therapy simulation)  (strategy: remove_loss_aversion)

CAUSAL CHAIN (integration.py)
  - [loss_aversion=N/A]  causal effect on decision: +0.0381  — Loss aversion weakly increases the probability of selling (effect = +0.038). Reducing it via reframing exercises could be a high-impact intervention.
  - [unrealized_gl=-0.15]  causal effect on decision: +0.8114  — Unrealized gain/loss strongly increases selling probability (effect = +0.811). Losses increase sell probability more than gains decrease it.

SHAP CONTRIBUTIONS (bias_detector.py)

SIMULATION (simulator.py)
  hold mean   :        148,823
  hold median :        139,712
  hold p10    :         81,882
  hold p90    :        230,245


**What is wrong with this**

- `causal effect on decision: +0.0412` — a number with no unit, no direction a person recognises, and no consequence
- `p10`, `p90` — percentile notation
- SHAP values explain *the model*, not *the decision*
- Nothing connects the bias to the money, and nothing says what to do

---
## 3. The explanation

Same inputs. Nothing recomputed. Only the presentation changes.

In [4]:
explainer   = DecisionExplainer(use_llm=False)   # deterministic template path
explanation = explainer.explain(bias_result, sim_result, user_input, lang="en")

for section in explanation["sections"]:
    display(Markdown(f"### {section['title']}\n\n{section['text']}"))

### In short

You're thinking about selling while your investment is down 15%. Before you do, here's what that one decision is likely to cost you over the next 14 years.

### What's going on right now

Your investment is down 15% right now, and that genuinely hurts. Here's the thing though: research on thousands of investors shows a loss feels about twice as painful as the same-sized gain feels good. So the urge you're feeling is stronger than the actual damage warrants.

That urge has a job — it wants you to stop the pain. But selling doesn't stop the loss. It makes it permanent. Right now the loss is only on paper, and paper losses have a habit of recovering when you give them time.

### Your two possible futures

We ran your actual situation through 1,000 different versions of the next 14 years — good markets, bad markets, crashes and recoveries. Here's where each choice tends to leave you.

**If you stay invested:** around ₹1.4 lakh by year 14. If things go badly, closer to ₹81,882. If things go well, about ₹2.3 lakh.

**If you sell now:** around ₹1.3 lakh. If things go badly, closer to ₹1 lakh. If things go well, about ₹1.6 lakh.

Out of every 100 possible futures, staying invested comes out ahead in 55 of them. Selling now would leave you about ₹9,540 poorer on average — that's 13% of what you have today.

### The story of your money

**Where your money stands today.** You have ₹75,000 invested, and it is currently worth ₹63,750.

If you sell today you don't walk away with ₹63,750. After brokerage, taxes and the gap between the buying and selling price — roughly 5% in all — you'd be left with ₹61,097 to put somewhere else. So selling costs you ₹2,653 straight away, before your money has had the chance to earn anything.

**What each choice then does with it.**

- **If you stay invested** — your ₹63,750 stays in your current mix: 56% in shares and 44% in bonds. That split comes from your age, your 14-year timeline, and how much risk you're comfortable with. Shares jump around a lot, but over long stretches they are what actually grows your money.
- **If you sell now** — your ₹61,097 goes somewhere much safer: 17% in shares and 83% in bonds. It stops giving you sleepless nights, but bonds earn only about 5.5% a year, so your money grows far more slowly.

That second point is the one almost everybody misses. Selling isn't just leaving this one investment. After a fright, people put the money somewhere safe and then leave it there — often for years. Over 14 years, that caution usually costs more than the fall you were trying to escape.

**Year by year.** Here is where your money is most likely to sit at the end of each year. The last column is how much more you'd have by staying invested.

| Year | If you stay invested | If you sell now | Difference |
|---|---|---|---|
| Today | ₹63,750 | ₹61,097 | +₹2,653 |
| 1 | ₹68,614 | ₹64,915 | +₹3,699 |
| 2 | ₹74,180 | ₹68,725 | +₹5,455 |
| 3 | ₹80,039 | ₹73,087 | +₹6,952 |
| 4 | ₹85,911 | ₹77,440 | +₹8,471 |
| 5 | ₹92,062 | ₹82,337 | +₹9,726 |
| 6 | ₹99,937 | ₹87,003 | +₹12,934 |
| 7 | ₹1,06,782 | ₹92,104 | +₹14,679 |
| 8 | ₹1,15,277 | ₹97,332 | +₹17,945 |
| 9 | ₹1,22,980 | ₹1,03,409 | +₹19,571 |
| 10 | ₹1,33,612 | ₹1,09,615 | +₹23,996 |
| 11 | ₹1,12,483 | ₹1,09,347 | +₹3,136 |
| 12 | ₹1,20,237 | ₹1,16,007 | +₹4,230 |
| 13 | ₹1,29,635 | ₹1,23,424 | +₹6,211 |
| 14 | ₹1,39,712 | ₹1,30,172 | +₹9,540 |

You'll notice both columns fall between year 10 and year 11. That's a market crash in this particular future — the kind that happens every decade or so. Staying invested falls hardest, which is exactly the moment selling feels like the obvious thing to do. It is also the moment the recovery afterwards does the most work for you.

**How these numbers were worked out.** Each year, your money is multiplied by what your mix earned that year — partly how shares did, partly the interest from bonds. The next year then starts from that new total, so growth builds on earlier growth. That is also why the difference between the two columns doesn't grow by the same amount every year — it widens through good stretches and narrows through bad ones.

Nobody knows which way markets will actually go, so we didn't guess. We ran your situation through 1,000 different versions of the next 14 years — booms, slumps and crashes included — and the table shows the middle one. Half of those futures turned out better than this, half turned out worse.

The difference grows steadily for years, then narrows near the end when a crash pulls the invested column down. That is worth seeing plainly: staying invested does not win smoothly, and it does not win in every future. It still ends ₹9,540 ahead here, and it comes out ahead in 55 of every 100 futures we ran.

### Why we're telling you this

We looked at what actually drove this decision, and the biggest factor was how strongly losses affect you. After that came how the market has moved recently, then how far your investment is down right now.

That's the tell. A decision driven by how much a loss stings, rather than by anything that changed about the investment itself, is usually your emotions talking — not your judgement.

### What to do now

**Our suggestion: stay invested. Don't sell today.**

You have about 14 years before you need this money, which isn't long. With a shorter runway there's less time to recover from a bad patch, so protecting what you have matters more than chasing growth. You already have emergency savings set aside, so you're not being forced to sell to cover anything — this is a choice, not a necessity.

How sure are we? We're fairly sure about this read of your situation (90%). The rupee figures are estimates drawn from 1,000 simulated futures — they show you what usually happens, not what will happen. Nobody can promise you the future, and you should be suspicious of anyone who does.

### The same explanation in Hindi

Hindi is not a translation of the English output — it is a parallel, hand-written template
bank, so it works offline with no LLM and no translation service.

In [5]:
hindi = explainer.explain(bias_result, sim_result, user_input, lang="hi")

for section in hindi["sections"]:
    display(Markdown(f"### {section['title']}\n\n{section['text']}"))

### संक्षेप में

आपका निवेश 15% नीचे है और आप बेचने की सोच रहे हैं। बेचने से पहले देखिए कि अगले 14 साल में इस एक फ़ैसले की कीमत क्या हो सकती है।

### अभी क्या हो रहा है

अभी आपका निवेश 15% नीचे है, और यह वाक़ई तकलीफ़ देता है। पर बात यह है कि हज़ारों निवेशकों पर हुए शोध बताते हैं — नुकसान का दर्द उतने ही बड़े फ़ायदे की ख़ुशी से लगभग दोगुना महसूस होता है। यानी जो बेचैनी आप महसूस कर रहे हैं, वह असली नुकसान से कहीं ज़्यादा बड़ी है।

इस बेचैनी का एक ही मक़सद है — दर्द रोकना। लेकिन बेचने से नुकसान रुकता नहीं, वह पक्का हो जाता है। अभी यह नुकसान सिर्फ़ काग़ज़ पर है, और काग़ज़ का नुकसान समय मिलने पर अक्सर वापस भर जाता है।

### आपके दो संभावित भविष्य

हमने आपकी असली स्थिति को अगले 14 साल के 1,000 अलग-अलग रूपों से गुज़ारा — अच्छे बाज़ार, बुरे बाज़ार, गिरावटें और वापसियाँ। हर विकल्प आपको आमतौर पर कहाँ ले जाता है, यह रहा।

**अगर आप निवेश बनाए रखते हैं:** 14वें साल तक लगभग ₹1.4 लाख। अगर हालात बुरे रहे तो क़रीब ₹81,882। अगर हालात अच्छे रहे तो लगभग ₹2.3 लाख।

**अगर आप अभी बेच देते हैं:** लगभग ₹1.3 लाख। अगर हालात बुरे रहे तो क़रीब ₹1 लाख। अगर हालात अच्छे रहे तो लगभग ₹1.6 लाख।

हर 100 संभावित भविष्य में से 55 में निवेश बनाए रखना आगे रहता है। अभी बेचने पर औसतन लगभग ₹9,540 कम रह जाएगा — यानी आज आपके पास जो है उसका 13%।

### आपके पैसे की कहानी

**आज आपका पैसा कहाँ खड़ा है।** आपने ₹75,000 निवेश किए हैं, और उनकी मौजूदा क़ीमत ₹63,750 है।

अगर आप आज बेचते हैं तो आपके हाथ में ₹63,750 नहीं आएँगे। ब्रोकरेज, टैक्स और ख़रीद-बिक्री की क़ीमत के अंतर को मिलाकर लगभग 5% कट जाता है — यानी आपके पास कहीं और लगाने के लिए ₹61,097 बचेंगे। इस तरह बेचने की क़ीमत आपको तुरंत ₹2,653 पड़ती है, इससे पहले कि आपके पैसे को कुछ कमाने का मौक़ा भी मिले।

**इसके बाद हर विकल्प आपके पैसे के साथ क्या करता है।**

- **अगर आप निवेश बनाए रखते हैं** — आपके ₹63,750 आपके मौजूदा मिश्रण में ही रहते हैं: 56% शेयरों में और 44% बॉन्ड में। यह बँटवारा आपकी उम्र, आपके 14 साल के समय और आपकी जोखिम सहने की क्षमता से तय होता है। शेयर बहुत ऊपर-नीचे होते हैं, लेकिन लंबे समय में असल बढ़त वही देते हैं।
- **अगर आप अभी बेच देते हैं** — आपके ₹61,097 कहीं ज़्यादा सुरक्षित जगह चले जाते हैं: 17% शेयरों में और 83% बॉन्ड में। रातों की नींद लौट आती है, पर बॉन्ड साल में सिर्फ़ लगभग 5.5% कमाते हैं, इसलिए पैसा बहुत धीरे बढ़ता है।

दूसरी बात वही है जो लगभग सबसे छूट जाती है। बेचने का मतलब सिर्फ़ इस एक निवेश से निकलना नहीं होता। एक बार डर बैठ जाने के बाद लोग पैसा किसी सुरक्षित जगह रख देते हैं और फिर उसे वहीं छोड़ देते हैं — अक्सर सालों तक। 14 साल में यह सावधानी आमतौर पर उस गिरावट से भी महँगी पड़ती है जिससे आप बचना चाहते थे।

**साल दर साल।** नीचे देखिए कि हर साल के अंत में आपका पैसा सबसे संभावित रूप से कहाँ होगा। आख़िरी ख़ाना बताता है कि निवेश बनाए रखने पर आपके पास कितना ज़्यादा होता।

| साल | निवेश बनाए रखने पर | अभी बेच देने पर | अंतर |
|---|---|---|---|
| आज | ₹63,750 | ₹61,097 | +₹2,653 |
| 1 | ₹68,614 | ₹64,915 | +₹3,699 |
| 2 | ₹74,180 | ₹68,725 | +₹5,455 |
| 3 | ₹80,039 | ₹73,087 | +₹6,952 |
| 4 | ₹85,911 | ₹77,440 | +₹8,471 |
| 5 | ₹92,062 | ₹82,337 | +₹9,726 |
| 6 | ₹99,937 | ₹87,003 | +₹12,934 |
| 7 | ₹1,06,782 | ₹92,104 | +₹14,679 |
| 8 | ₹1,15,277 | ₹97,332 | +₹17,945 |
| 9 | ₹1,22,980 | ₹1,03,409 | +₹19,571 |
| 10 | ₹1,33,612 | ₹1,09,615 | +₹23,996 |
| 11 | ₹1,12,483 | ₹1,09,347 | +₹3,136 |
| 12 | ₹1,20,237 | ₹1,16,007 | +₹4,230 |
| 13 | ₹1,29,635 | ₹1,23,424 | +₹6,211 |
| 14 | ₹1,39,712 | ₹1,30,172 | +₹9,540 |

ध्यान दीजिए कि साल 10 और साल 11 के बीच दोनों ख़ाने नीचे गिरते हैं। इस ख़ास भविष्य में वहाँ बाज़ार में बड़ी गिरावट आती है — ऐसी गिरावट लगभग हर दशक में आती है। निवेश बनाए रखने वाला ख़ाना सबसे ज़्यादा गिरता है, और ठीक वही पल होता है जब बेचना सबसे सही लगता है। वही पल यह भी तय करता है कि उसके बाद की वापसी आपके लिए कितना काम करेगी।

**ये आँकड़े कैसे निकाले गए।** हर साल आपका पैसा उस साल की कमाई से गुणा होता है — कुछ हिस्सा शेयरों के प्रदर्शन से, कुछ बॉन्ड के ब्याज से। अगला साल उसी नए कुल से शुरू होता है, यानी बढ़त पर फिर बढ़त मिलती है। इसी वजह से दोनों ख़ानों का अंतर हर साल बराबर नहीं बढ़ता — अच्छे दौर में चौड़ा होता है और बुरे दौर में सिमट जाता है।

बाज़ार किस ओर जाएगा यह कोई नहीं जानता, इसलिए हमने अंदाज़ा नहीं लगाया। हमने आपकी स्थिति को अगले 14 साल के 1,000 अलग-अलग रूपों से गुज़ारा — तेज़ी, मंदी और बड़ी गिरावटें भी शामिल — और तालिका बीच वाला रूप दिखाती है। उनमें से आधे भविष्य इससे बेहतर रहे और आधे इससे ख़राब।

यह अंतर सालों तक लगातार बढ़ता है, फिर आख़िर में एक गिरावट निवेश वाले ख़ाने को नीचे खींच देती है और अंतर सिमट जाता है। इसे साफ़-साफ़ देख लेना ज़रूरी है: निवेश बनाए रखना हर साल आराम से नहीं जीतता, और हर भविष्य में नहीं जीतता। फिर भी यहाँ यह ₹9,540 आगे रहता है, और हमने जो 100 भविष्य चलाए उनमें से 55 में यह आगे रहता है।

### हम यह क्यों कह रहे हैं

हमने देखा कि इस फ़ैसले के पीछे असल में क्या था, और सबसे बड़ी वजह थी नुकसान आप पर कितना असर करता है। इसके बाद आया बाज़ार हाल में कैसा चला है, फिर आपका निवेश अभी कितना नीचे है।

यही पहचान है। जब फ़ैसला निवेश में हुए किसी असली बदलाव से नहीं, बल्कि नुकसान की चुभन से निकले, तो वह आमतौर पर आपकी भावनाएँ बोल रही होती हैं — आपकी समझ नहीं।

### अब क्या करें

**हमारी सलाह: निवेश बनाए रखिए। आज बेचिए मत।**

इस पैसे की ज़रूरत पड़ने में आपके पास लगभग 14 साल हैं, जो ज़्यादा नहीं है। कम समय में बुरे दौर से उबरने का मौक़ा कम मिलता है, इसलिए यहाँ बढ़त के पीछे भागने से ज़्यादा ज़रूरी है जो है उसे बचाकर रखना। आपके पास आपातकालीन बचत पहले से मौजूद है, इसलिए किसी ख़र्च के लिए आपको बेचना नहीं पड़ रहा — यह आपकी पसंद है, मजबूरी नहीं।

हमें कितना यक़ीन है? आपकी स्थिति के इस आकलन पर हमें काफ़ी हद तक यक़ीन है (90%)। रुपयों के ये आँकड़े 1,000 संभावित भविष्यों से निकाले गए अनुमान हैं — ये बताते हैं कि आमतौर पर क्या होता है, यह नहीं कि क्या होगा ही। भविष्य का वादा कोई नहीं कर सकता, और जो करे उस पर शक कीजिए।

---
## 4. All four bias narratives

Each detected bias gets its own explanation of what is happening and why — the psychology
differs, so the words must differ too.

In [6]:
for bias in ("loss_aversion", "herd", "disposition", "rational"):
    # Hold the simulation fixed and vary only the detected bias.
    case_bias = {**bias_result, "bias_type": bias, "is_biased": bias != "rational",
                 "confidence": 0.82}
    case_input = {**user_input,
                  "unrealized_gl": 0.18 if bias == "disposition" else -0.15}
    for lang in ("en", "hi"):
        exp = explainer.explain(case_bias, sim_result, case_input, lang)
        display(Markdown(f"**{bias} / {T.LANG_NAMES[lang]}** — {exp['headline']}"))

**loss_aversion / English** — You're thinking about selling while your investment is down 15%. Before you do, here's what that one decision is likely to cost you over the next 14 years.

**loss_aversion / हिंदी** — आपका निवेश 15% नीचे है और आप बेचने की सोच रहे हैं। बेचने से पहले देखिए कि अगले 14 साल में इस एक फ़ैसले की कीमत क्या हो सकती है।

**herd / English** — Roughly 72% of investors are selling right now, and you're about to do the same. Here's what following them is likely to cost you.

**herd / हिंदी** — अभी लगभग 72% निवेशक बेच रहे हैं, और आप भी वही करने जा रहे हैं। देखिए कि उनके पीछे चलने की कीमत क्या हो सकती है।

**disposition / English** — You're up 18% and thinking about taking the money off the table. Locking in a win feels great — here's what it tends to cost.

**disposition / हिंदी** — आप 18% ऊपर हैं और मुनाफ़ा निकाल लेने की सोच रहे हैं। जीत पक्की कर लेना अच्छा लगता है — देखिए इसकी कीमत आमतौर पर क्या होती है।

**rational / English** — This looks like a clear-headed decision. We found no sign that fear, the crowd, or habit is steering you — so here's the straightforward money picture.

**rational / हिंदी** — यह एक समझदारी भरा फ़ैसला लगता है। हमें ऐसा कोई संकेत नहीं मिला कि डर, भीड़ या पुरानी आदत आपको चला रही हो — तो सीधे पैसे की तस्वीर देखते हैं।

### Bias resisted

When a user is susceptible to a bias but their stated intent goes *against* it, the system
should recognise that rather than warn them about a decision they are already getting right.

In [7]:
resisted = explainer.explain(
    {**bias_result, "bias_type": "herd", "is_biased": True},
    sim_result,
    {**user_input, "decision_intent": "Planning to hold"},
    "en")

print("resisted:", resisted["resisted"])
display(Markdown(resisted["headline"]))
display(Markdown(f"**{resisted['recommendation']}**"))

resisted: True


Roughly 72% of investors are selling right now, and you've decided not to join them. Sitting still while everyone runs is genuinely hard — and it's the right call here.

**Our suggestion: stick with your decision to stay invested. You're on the right track.**

---
## 5. The jargon translation layer

This table is the mechanism. Everything on the left is what the models produce; everything
on the right is what the user reads.

In [8]:
import pandas as pd

rows = [{"Internal term": k, "Shown to the user (EN)": v,
         "Shown to the user (HI)": T.JARGON_MAP["hi"].get(k, "")}
        for k, v in T.JARGON_MAP["en"].items()]
pd.set_option("display.max_colwidth", 70)
display(pd.DataFrame(rows))

print("Blocked terms:", len(T.JARGON_BLOCKLIST))
print("Jargon found in this explanation:",
      find_jargon(" ".join(explanation["narrative"].values())) or "none")

,Internal term,Shown to the user (EN),Shown to the user (HI)
0,unrealized_gl,how much your investment is up or down right now,आपका निवेश अभी कितना ऊपर या नीचे है
1,p10,if things go badly,अगर हालात बुरे रहे
2,p90,if things go well,अगर हालात अच्छे रहे
3,p50,"the middle, most likely case","बीच का, सबसे संभावित हाल"
4,median,"the middle, most likely case","बीच का, सबसे संभावित हाल"
5,opportunity_cost,money you'd miss out on by selling,बेचने पर जो पैसा आपके हाथ से निकल जाएगा
6,prob_hold_better,how often staying invested wins,निवेश बनाए रखना कितनी बार बेहतर रहता है
7,disposition effect,selling winners too early and holding losers too long,फ़ायदे वाले निवेश जल्दी बेच देना और घाटे वाले पकड़े रहना
8,loss aversion,losses hurting about twice as much as gains feel good,नुकसान का दर्द बराबर के फ़ायदे की ख़ुशी से लगभग दोगुना लगना
9,herd behaviour,following what everyone else is doing,जो सब कर रहे हैं वही करना


Blocked terms: 30
Jargon found in this explanation: none


### Indian money formatting

`Rs.1,23,45,678` is technically precise and practically unreadable. Amounts are rounded to
the units people actually speak in.

In [9]:
for value in (850, 45_600, 1_20_000, 24_50_000, 3_12_00_000):
    print(f"{value:>12,}  ->  {T.format_inr(value, 'en'):<16} {T.format_inr(value, 'hi')}")

         850  ->  ₹850             ₹850
      45,600  ->  ₹45,600          ₹45,600
     120,000  ->  ₹1.2 lakh        ₹1.2 लाख
   2,450,000  ->  ₹24.5 lakh       ₹24.5 लाख
  31,200,000  ->  ₹3.12 crore      ₹3.12 करोड़


---
## 6. Validation metrics

Five metrics with stated targets, measured over a sample of real pipeline runs plus one
case per bias narrative. Generated by:

```
python scripts/evaluate_explanations.py --n-users 28
```

In [10]:
report_path = os.path.join("..", "results", "reports", "explanation_quality.json")

if os.path.exists(report_path):
    quality = json.load(open(report_path, encoding="utf-8"))
    print(f"Sample: {quality['n_users']} live decisions "
          f"+ {quality.get('n_template_coverage_cases', 0)} coverage cases")
    print(f"Bias coverage: {quality['bias_coverage']}\n")

    table = []
    for lang, summary in quality["by_language"].items():
        table.append({
            "Language"       : T.LANG_NAMES[lang],
            "Explanations"   : summary["n_explanations"],
            "Complete"       : f"{summary['completeness_pct']:.0f}%",
            "Readable"       : f"{summary['readability_pass_pct']:.0f}%",
            "Actionable"     : f"{summary['actionability_pct']:.0f}%",
            "Personalised"   : f"{summary['personalisation_pct']:.0f}%",
            "Jargon leaks"   : summary["jargon_leaks"],
            "Mean Flesch"    : summary.get("mean_flesch") or "n/a (Devanagari)",
            "Words/sentence" : summary["mean_words_per_sentence"],
            "All targets met": "YES" if summary["all_targets_met"] else "NO",
        })
    display(pd.DataFrame(table).set_index("Language").T)
else:
    print("Run scripts/evaluate_explanations.py first.")

Sample: 28 live decisions + 4 coverage cases
Bias coverage: {'disposition': 2, 'herd': 21, 'loss_aversion': 8, 'rational': 1}



Language,English,हिंदी
Explanations,32,32
Complete,100%,100%
Readable,100%,100%
Actionable,100%,100%
Personalised,100%,100%
Jargon leaks,0,0
Mean Flesch,78.3,n/a (Devanagari)
Words/sentence,10.9,12.2
All targets met,YES,YES


**Reading the table**

- **Flesch Reading Ease ≥ 60** is roughly an 8th-grade reading level. Flesch is undefined for
  Devanagari — its syllable model is built for English — so Hindi is bounded by sentence
  length (≤ 20 words) instead.
- **Jargon leaks = 0** is enforced by a blocklist scan, and by a test in
  `tests/test_explainer.py` that fails the build if a technical term reaches the user.

---
## 7. The downloadable report

The user can keep the explanation, print it, or show it to an advisor.

| Format | English | Hindi | Note |
|---|---|---|---|
| Markdown | yes | yes | plain text |
| HTML | yes | yes | styled, print-ready — **recommended** |
| PDF | yes | no | see below |

**Why no Hindi PDF.** ReportLab has no OpenType text shaper. Devanagari needs one: in
`कि` the matra `ि` is stored after the consonant but must be drawn *before* it, and
conjuncts like `क्ष` must be composed. Without shaping, ReportLab draws codepoints in
storage order and the output is wrong. Refusing is the correct behaviour; the HTML report
prints to a correct PDF from any browser, which shapes Devanagari properly.

In [11]:
full = generate_complete_explanation(bias_result, sim_result, user_input, "en", use_llm=False)

print(render_markdown(full)[:900], "...\n")

pdf_bytes = report_mod.to_pdf(full)
print("English PDF :", len(pdf_bytes), "bytes,", report_mod.filename_for(full, "pdf"))

hindi_full = generate_complete_explanation(bias_result, sim_result, user_input, "hi", use_llm=False)
try:
    report_mod.to_pdf(hindi_full)
except ValueError as exc:
    print("Hindi PDF   : refused —", exc)

# Your Decision Explained

## In short

You're thinking about selling while your investment is down 15%. Before you do, here's what that one decision is likely to cost you over the next 14 years.

## What's going on right now

Your investment is down 15% right now, and that genuinely hurts. Here's the thing though: research on thousands of investors shows a loss feels about twice as painful as the same-sized gain feels good. So the urge you're feeling is stronger than the actual damage warrants.

That urge has a job — it wants you to stop the pain. But selling doesn't stop the loss. It makes it permanent. Right now the loss is only on paper, and paper losses have a habit of recovering when you give them time.

## Your two possible futures

We ran your actual situation through 1,000 different versions of the next 14 years — good markets, bad markets, crashes and recoveries. Here's where e ...

English PDF : 7879 bytes, decision_explained_loss_aversion_en.pdf
Hindi PDF   : refused — हिंद

In [12]:
# The HTML report, rendered inline. Devanagari shapes correctly here because the
# browser does the shaping.
display(HTML(report_mod.to_html(hindi_full)))

साल,निवेश बनाए रखने पर,अभी बेच देने पर,अंतर
आज,"₹63,750","₹61,097","+₹2,653"
1,"₹68,614","₹64,915","+₹3,699"
2,"₹74,180","₹68,725","+₹5,455"
3,"₹80,039","₹73,087","+₹6,952"
4,"₹85,911","₹77,440","+₹8,471"
5,"₹92,062","₹82,337","+₹9,726"
6,"₹99,937","₹87,003","+₹12,934"
7,"₹1,06,782","₹92,104","+₹14,679"
8,"₹1,15,277","₹97,332","+₹17,945"
9,"₹1,22,980","₹1,03,409","+₹19,571"


---
## 8. The optional LLM layer, and why it cannot invent a number

The templates are deterministic and always correct, but slightly stiff. An optional pass
(Gemini Flash free tier, off by default) can improve the wording — under a rule that is
*enforced*, not trusted:

> The model may reword. It may not introduce, remove, or alter a number.

Every numeric token and rupee amount in the polished text must already exist in the
template text. If one does not, the polish is discarded and the user reads the template
version. This is what makes it safe to let a language model near figures that drive real
money decisions.

In [13]:
import llm_polish

print("Configuration:", llm_polish.provider_info())
print()

original = "In 20 years you would have about Rs.5.8 lakh, and you are down 15%."

tests = {
    "pure rewording (accepted)"  : "You'd have roughly Rs.5.8 lakh in 20 years, and you're down 15%.",
    "invented amount (rejected)" : "In 20 years you would have about Rs.9.9 lakh, and you are down 15%.",
    "altered percentage (rejected)": "In 20 years you would have about Rs.5.8 lakh, and you are down 45%.",
    "truncated (rejected)"       : "You are down.",
}
for label, candidate in tests.items():
    print(f"  {llm_polish.guard_ok(original, candidate)!s:<5}  {label}")

LLM polish rejected — invented numbers: ['9.9']


LLM polish rejected — invented numbers: ['45']


LLM polish rejected — length drift x0.19


Configuration: {'enabled': False, 'provider': 'gemini', 'model': 'gemini-flash-latest', 'key_set': False}

  True   pure rewording (accepted)
  False  invented amount (rejected)
  False  altered percentage (rejected)
  False  truncated (rejected)


---
## Summary

| Delivered | Where |
|---|---|
| Template bank, English + Hindi | `src/explanation_templates.py` |
| Explainer engine | `src/explainer.py` |
| Guarded optional LLM layer | `src/llm_polish.py` |
| Downloadable reports (md / html / pdf) | `src/explanation_report.py` |
| `POST /explain`, `/explain/report`, `/explain/status` | `src/api.py` |
| Narrative + language toggle + downloads in the UI | `src/dashboard.py` |
| Validation metrics | `scripts/evaluate_explanations.py` |
| Tests | `tests/test_explainer.py` |

**What changed for the user.** Before: `causal effect on decision: +0.0412`. After: a story
that says what they are about to do, why their brain is pushing them there, what it costs in
rupees over their own horizon, and what to do now — in the language they read.